INTEGRACION DE DATOS

In [1]:
from google.colab import drive
import os

# Montar el drive
drive.mount('/content/drive')

Mounted at /content/drive


EDAFOLOGÍA SINCRONIZADA

In [2]:
import os
from osgeo import gdal

# 1. Configuración de rutas según tus carpetas en Drive
ruta_terreno = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Terrain Model/'
# Carpeta donde están los archivos de la imagen que enviaste
ruta_edafologia_raw = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/'
# Carpeta de destino final para E3
ruta_e3_integration = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/'

if not os.path.exists(ruta_e3_integration):
    os.makedirs(ruta_e3_integration)

# Definición de archivos
template_path = os.path.join(ruta_terreno, 'Mosaico_Maestro_CEM_1.5m.tif')
edafologia_in = os.path.join(ruta_edafologia_raw, 'e1403.tif')
edafologia_out = os.path.join(ruta_e3_integration, 'Edafologia_Sincronizada_1.5m.tif')

def align_layer(input_file, output_file, template_file):
    # Abrir la plantilla para obtener dimensiones y coordenadas exactas
    temp_ds = gdal.Open(template_file)
    projection = temp_ds.GetProjection()
    geotransform = temp_ds.GetGeoTransform()
    width = temp_ds.RasterXSize
    height = temp_ds.RasterYSize

    # Configurar opciones de Warp para alineación perfecta al Master Grid
    options = gdal.WarpOptions(
        format='GTiff',
        outputBounds=[geotransform[0], geotransform[3] + geotransform[5]*height,
                      geotransform[0] + geotransform[1]*width, geotransform[3]],
        xRes=geotransform[1],
        yRes=geotransform[5],
        dstSRS=projection,
        resampleAlg=gdal.GRA_NearestNeighbour, # Crucial para no promediar tipos de suelo
        dstNodata=-9999,
        creationOptions=['COMPRESS=DEFLATE', 'TILED=YES']
    )

    print(f"Sincronizando capa: {os.path.basename(input_file)}...")
    gdal.Warp(output_file, input_file, options=options)

# Ejecución del proceso
try:
    if os.path.exists(edafologia_in):
        align_layer(edafologia_in, edafologia_out, template_path)
        print(f"✅ ¡Éxito! Edafología sincronizada guardada en: {edafologia_out}")
    else:
        print(f"❌ Error: No se encontró el archivo e1403.tif en {ruta_edafologia_raw}")
except Exception as e:
    print(f"❌ Error durante el procesamiento: {e}")

/usr/local/lib/python3.12/dist-packages/osgeo/gdal.py:312: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Sincronizando capa: e1403.tif...
✅ ¡Éxito! Edafología sincronizada guardada en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/Edafologia_Sincronizada_1.5m.tif


In [ ]:
RIESGOS

In [3]:
import os
import rasterio
import numpy as np

# Rutas de insumos
ruta_terreno = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Terrain Model/'
ruta_integration = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/'

dtm_path = os.path.join(ruta_terreno, 'Mosaico_Maestro_CEM_1.5m.tif')
urbana_path = os.path.join(ruta_integration, 'Mascara_Urbana_Refinamiento.tif')
output_riesgo = os.path.join(ruta_integration, 'Mapa_Inundacion_Umbral_3msnm.tif')

with rasterio.open(dtm_path) as dtm_src, rasterio.open(urbana_path) as urb_src:
    dtm = dtm_src.read(1)
    urbana = urb_src.read(1)

    # Lógica: Elevación <= 3.0 Y está en zona urbana (ignorando NoData)
    riesgo = np.where((dtm <= 3.0) & (dtm > -1.5) & (urbana > 0), 1, 0).astype('uint8')

    # Guardar con los metadatos del Master Grid
    meta = dtm_src.meta.copy()
    meta.update(dtype='uint8', count=1, nodata=0)

    with rasterio.open(output_riesgo, 'w', **meta) as dst:
        dst.write(riesgo, 1)

print(f"✅ Mapa de Riesgo generado: {output_riesgo}")

✅ Mapa de Riesgo generado: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/Mapa_Inundacion_Umbral_3msnm.tif


INFRAESTRUCTURA VULNERABLE

In [8]:
import os
import rasterio
import numpy as np

# 1. Definición de rutas consistentes con tu estructura de Drive
ruta_integration = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/'

# Definir archivos de entrada basándose en tu captura de pantalla de E3
output_riesgo = os.path.join(ruta_integration, 'Mapa_Inundacion_Umbral_3msnm.tif')
vias_path = os.path.join(ruta_integration, 'Vialidades_Principales_Segmentadas.tif')

# Archivo de salida
output_exposicion = os.path.join(ruta_integration, 'Infraestructura_Vulnerable_3msnm.tif')

# 2. Validación y Ejecución
if os.path.exists(output_riesgo) and os.path.exists(vias_path):
    print("Archivos detectados. Iniciando cruce de infraestructura...")

    with rasterio.open(output_riesgo) as riesgo_src, rasterio.open(vias_path) as vias_src:
        riesgo = riesgo_src.read(1)
        vias = vias_src.read(1)

        # Intersección: Zona de riesgo (1) Y presencia de vialidad (> 0)
        # Aseguramos que el riesgo sea exactamente el umbral de 3msnm calculado
        vias_vulnerables = np.where((riesgo == 1) & (vias > 0), 1, 0).astype('uint8')

        # Usamos los metadatos de las vialidades para mantener la coherencia espacial
        meta = vias_src.meta.copy()
        meta.update(dtype='uint8', count=1, nodata=0)

        with rasterio.open(output_exposicion, 'w', **meta) as dst:
            dst.write(vias_vulnerables, 1)

    print(f"✅ ¡Éxito! Mapa de Infraestructura Vulnerable generado en: {output_exposicion}")
else:
    print("❌ ERROR: No se encontró uno de los archivos en la ruta especificada.")
    print(f"Buscando riesgo en: {output_riesgo}")
    print(f"Buscando vialidades en: {vias_path}")

Archivos detectados. Iniciando cruce de infraestructura...
✅ ¡Éxito! Mapa de Infraestructura Vulnerable generado en: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/Infraestructura_Vulnerable_3msnm.tif


DATASET MAESTRO MULTICANAL


In [9]:
import os
from osgeo import gdal

# 1. Rutas de los componentes ya validados en tu Drive
ruta_terreno = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E2/Terrain Model/'
ruta_integration = '/content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/'

# Definición de las 4 bandas
b1_elevacion = os.path.join(ruta_terreno, 'Mosaico_Maestro_CEM_1.5m.tif')
b2_edafologia = os.path.join(ruta_integration, 'Edafologia_Sincronizada_1.5m.tif')
b3_infraestructura = os.path.join(ruta_integration, 'Vialidades_Principales_Segmentadas.tif')
b4_urbana = os.path.join(ruta_integration, 'Mascara_Urbana_Refinamiento.tif')

# Archivo de salida final
output_master = os.path.join(ruta_integration, 'Dataset_Maestro_Multicanal_Veracruz.tif')

# 2. Lista ordenada de archivos (El orden aquí define el número de banda)
input_list = [b1_elevacion, b2_edafologia, b3_infraestructura, b4_urbana]

print("Iniciando el apilamiento (stacking) de bandas...")

# 3. Crear el dataset multibanda
# Usamos -separate para que cada archivo de entrada sea una banda distinta
vrt_options = gdal.BuildVRTOptions(separate=True)
vrt = gdal.BuildVRT('/content/temp_master.vrt', input_list, options=vrt_options)

# 4. Convertir a GeoTIFF final optimizado para Deep Learning
# TILED=YES e INTERLEAVE=BAND facilitan la lectura aleatoria de ventanas (chips) en PyTorch
creation_options = [
    'COMPRESS=DEFLATE',
    'TILED=YES',
    'INTERLEAVE=BAND',
    'BIGTIFF=YES'
]

try:
    gdal.Translate(output_master, vrt, format='GTiff', creationOptions=creation_options)
    print(f"✅ ¡Dataset Maestro generado exitosamente!")
    print(f"Ubicación: {output_master}")
except Exception as e:
    print(f"❌ Error al generar el dataset: {e}")

Iniciando el apilamiento (stacking) de bandas...
✅ ¡Dataset Maestro generado exitosamente!
Ubicación: /content/drive/MyDrive/DOCTORADO 2026/VISION ARTIFICIAL/E3/Data Integration/Dataset_Maestro_Multicanal_Veracruz.tif
